<a href="https://colab.research.google.com/github/Shamit-butterchicken/Bach-Cello-Suite-Music-Generator/blob/main/bach_music_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, losses, optimizers, callbacks
import music21

repo_dir = "/content/Generative_Deep_Learning_2nd_Edition/notebooks/11_music/01_transformer"
if os.path.exists(repo_dir):
    os.chdir(repo_dir)

from transformer_utils import load_parsed_files, get_midi_note, SinePositionEncoding

CUSTOM_START_NOTE = "C3"
CUSTOM_START_DURATION = "0.25"
EMBEDDING_DIM, KEY_DIM, N_HEADS, DROPOUT_RATE, FF_DIM = 256, 256, 5, 0.3, 256
EPOCHS, BATCH_SIZE, GENERATE_LEN = 150, 256, 50

notes, durations = load_parsed_files("./parsed_data/")

def create_dataset(elements):
    ds = tf.data.Dataset.from_tensor_slices(elements).batch(BATCH_SIZE, drop_remainder=True)
    vec = layers.TextVectorization(standardize=None, output_mode="int")
    vec.adapt(ds)
    return ds, vec, vec.get_vocabulary()

notes_ds, notes_vec, notes_vocab = create_dataset(notes)
durs_ds, durs_vec, durs_vocab = create_dataset(durations)

def prepare_inputs(n, d):
    tn, td = notes_vec(tf.expand_dims(n, -1)), durs_vec(tf.expand_dims(d, -1))
    return (tn[:, :-1], td[:, :-1]), (tn[:, 1:], td[:, 1:])

ds = tf.data.Dataset.zip((notes_ds, durs_ds)).map(prepare_inputs).cache().shuffle(100).prefetch(tf.data.AUTOTUNE)

def causal_mask(bs, nd, ns, dtype):
    m = tf.range(nd)[:, None] >= tf.range(ns) - ns + nd
    return tf.tile(tf.reshape(tf.cast(m, dtype), [1, nd, ns]), tf.concat([tf.expand_dims(bs, -1), [1, 1]], 0))

class TransformerBlock(layers.Layer):
    def __init__(self, heads, kdim, edim, fdim, name, drop=DROPOUT_RATE):
        super().__init__(name=name)
        self.attn = layers.MultiHeadAttention(heads, kdim, output_shape=edim)
        self.d1, self.ln1 = layers.Dropout(drop), layers.LayerNormalization(epsilon=1e-6)
        self.f1, self.f2 = layers.Dense(fdim, activation="relu"), layers.Dense(edim)
        self.d2, self.ln2 = layers.Dropout(drop), layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        s = tf.shape(x)
        out1 = self.ln1(x + self.d1(self.attn(x, x, attention_mask=causal_mask(s[0], s[1], s[1], tf.bool))))
        return self.ln2(out1 + self.d2(self.f2(self.f1(out1))))

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, vsize, edim):
        super().__init__()
        self.emb = layers.Embedding(vsize, edim, embeddings_initializer="he_uniform")
        self.pos = SinePositionEncoding()

    def call(self, x):
        e = self.emb(x)
        return e + self.pos(e)

n_in, d_in = layers.Input(shape=(None,), dtype=tf.int32), layers.Input(shape=(None,), dtype=tf.int32)
emb = layers.Concatenate()([
    TokenAndPositionEmbedding(len(notes_vocab), EMBEDDING_DIM // 2)(n_in),
    TokenAndPositionEmbedding(len(durs_vocab), EMBEDDING_DIM // 2)(d_in),
])
x = TransformerBlock(N_HEADS, KEY_DIM, EMBEDDING_DIM, FF_DIM, name="attention")(emb)
model = models.Model(
    [n_in, d_in],
    [layers.Dense(len(notes_vocab), activation="softmax")(x), layers.Dense(len(durs_vocab), activation="softmax")(x)],
)
model.compile(optimizers.Adam(0.003), loss=[losses.SparseCategoricalCrossentropy(), losses.SparseCategoricalCrossentropy()])

class ProgressPrint(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 25 == 0:
            print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {logs['loss']:.4f}")

model.fit(ds, epochs=EPOCHS, verbose=0, callbacks=[ProgressPrint()])

n2i = {n: i for i, n in enumerate(notes_vocab)}
d2i = {d: i for i, d in enumerate(durs_vocab)}

def sample_from(probs, temp=0.5):
    p = probs ** (1 / temp)
    return np.random.choice(len(p), p=p / np.sum(p))

def generate_music(s_notes, s_durs, max_tok=GENERATE_LEN):
    nt, dt = [n2i.get(x, 1) for x in s_notes], [d2i.get(x, 1) for x in s_durs]
    stream = music21.stream.Stream([music21.clef.BassClef()])
    for n, d in zip(s_notes, s_durs):
        nn = get_midi_note(n, d)
        if nn is not None:
            stream.append(nn)
    while len(nt) < max_tok:
        np_pred, dp_pred = model.predict([np.array([nt]), np.array([dt])], verbose=0)
        while True:
            ni, di = sample_from(np_pred[0][-1]), sample_from(dp_pred[0][-1])
            if ni == 1 or di == 1:
                continue
            sn, sd = notes_vocab[ni], durs_vocab[di]
            nn = get_midi_note(sn, sd)
            if not (isinstance(nn, (music21.chord.Chord, music21.note.Note, music21.note.Rest)) and sd == "0.0"):
                break
        if nn is not None:
            stream.append(nn)
        nt.append(ni); dt.append(di); s_notes.append(sn); s_durs.append(sd)
        if sn == "START":
            break
    return list(zip(s_notes, s_durs)), stream.chordify()

def run_and_print(s_notes, s_durs, label):
    seq, stream = generate_music(s_notes.copy(), s_durs.copy())
    fn = f"./output/{label}.mid"
    stream.write("midi", fp=fn)
    print(f"\n=== {label.upper()} (Saved to {fn}) ===")
    for i, (n, d) in enumerate(seq):
        print(f"{i:02d}: Note={n:<8} Duration={d}")

run_and_print(["START"], ["0.0"], "1_original_start_seed")
run_and_print([CUSTOM_START_NOTE], [CUSTOM_START_DURATION], f"2_custom_{CUSTOM_START_NOTE}_seed")
print("ALL DONE!")

Epoch 25/150 - loss: 2.9204
Epoch 50/150 - loss: 2.5672
Epoch 75/150 - loss: 2.2800
Epoch 100/150 - loss: 2.1120
Epoch 125/150 - loss: 1.9854
Epoch 150/150 - loss: 1.8860

=== 1_ORIGINAL_START_SEED (Saved to ./output/1_original_start_seed.mid) ===
00: Note=START    Duration=0.0
01: Note=C:major  Duration=0.0
02: Note=3/4TS    Duration=0.0
03: Note=B-3      Duration=0.5
04: Note=B-3      Duration=0.5
05: Note=A3       Duration=0.5
06: Note=A3       Duration=0.5
07: Note=E3       Duration=0.5
08: Note=F3       Duration=0.5
09: Note=D3       Duration=0.5
10: Note=F3       Duration=0.5
11: Note=D3       Duration=0.5
12: Note=F3       Duration=0.5
13: Note=D3       Duration=0.5
14: Note=F3       Duration=0.5
15: Note=A2       Duration=0.5
16: Note=F2       Duration=0.5
17: Note=E2       Duration=0.5
18: Note=G2       Duration=0.5
19: Note=E3       Duration=0.5
20: Note=F3       Duration=0.5
21: Note=C3       Duration=0.5
22: Note=B2       Duration=0.5
23: Note=F3       Duration=0.5
24: Note

In [ ]:
import os, subprocess
import music21
from IPython.display import Audio, display

os.makedirs("/content/output", exist_ok=True)

# 1. Exact sequences generated by your trained model
seq_original = [
    ("START", 0.0), ("C:major", 0.0), ("3/4TS", 0.0),
    ("B-3", 0.5), ("B-3", 0.5), ("A3", 0.5), ("A3", 0.5), ("E3", 0.5),
    ("F3", 0.5), ("D3", 0.5), ("F3", 0.5), ("D3", 0.5), ("F3", 0.5),
    ("D3", 0.5), ("F3", 0.5), ("A2", 0.5), ("F2", 0.5), ("E2", 0.5),
    ("G2", 0.5), ("E3", 0.5), ("F3", 0.5), ("C3", 0.5), ("B2", 0.5),
    ("F3", 0.5), ("D3", 0.5), ("F3", 0.5), ("B2", 0.5), ("F3", 0.5),
    ("B2", 0.5), ("G2", 0.5), ("B2", 0.5), ("G2", 0.5), ("C2", 0.5),
    ("F3", 0.5), ("E3", 0.5), ("F3", 0.5), ("E3", 0.5), ("D3", 0.5),
    ("C3", 0.5), ("B2", 0.5), ("A2", 0.5), ("G2", 0.5), ("F2", 0.5),
    ("E2", 0.5), ("D2", 0.5), ("F2", 0.5), ("E2", 0.5), ("G2", 0.5),
    ("C3", 0.5), ("B2", 0.5),
]

seq_custom = [
    ("C3", 0.25), ("C3", 0.25), ("B2", 0.25), ("C3", 0.25), ("D3", 0.25),
    ("C3", 0.25), ("A2", 0.25), ("B2", 0.25), ("A2", 0.25), ("B2", 0.25),
    ("A2", 0.25), ("B2", 0.25), ("A2", 0.25), ("B2", 0.25), ("D3", 0.25),
    ("B2", 0.25), ("D2", 0.25), ("B2", 0.25), ("D3", 0.25), ("B2", 0.25),
    ("D3", 0.25), ("F3", 0.25), ("B2", 0.25), ("F3", 0.25), ("G2", 0.25),
    ("E3", 0.25), ("G2", 0.25), ("D3", 0.25), ("F3", 0.25), ("G2", 0.25),
    ("E3", 0.25), ("F3", 0.25), ("G2", 0.25), ("E3", 0.25), ("F3", 0.25),
    ("G3", 0.25), ("G2", 0.25), ("F3", 0.25), ("E3", 0.25), ("F3", 0.25),
    ("G2", 0.25), ("E3", 0.25), ("F3", 0.25), ("G2", 0.25), ("E3", 0.25),
    ("D3", 0.25), ("G2", 0.25), ("E3", 0.25), ("C4", 0.25), ("G2", 0.25),
]

def build_midi(seq, midi_path):
    s = music21.stream.Stream()
    s.append(music21.instrument.Violoncello())
    s.append(music21.clef.BassClef())
    for note_str, dur in seq:
        if note_str == "START":
            continue
        elif ":" in note_str:
            tonic, mode = note_str.split(":")
            s.append(music21.key.Key(tonic, mode))
        elif "TS" in note_str:
            s.append(music21.meter.TimeSignature(note_str.replace("TS", "")))
        else:
            s.append(music21.note.Note(note_str, quarterLength=dur))
    s.write("midi", fp=midi_path)

# 2. Install FluidSynth audio synthesizer
print("Setting up cello synthesizer (~15 seconds)...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"], check=True)
soundfont = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

def render_and_play(seq, name, title):
    midi_path = f"/content/output/{name}.mid"
    wav_path = f"/content/output/{name}.wav"
    build_midi(seq, midi_path)
    subprocess.run(
        ["fluidsynth", "-ni", soundfont, midi_path, "-F", wav_path, "-r", "44100"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
    )
    print(f"\n{title}")
    display(Audio(wav_path))

render_and_play(seq_original, "1_original_start_seed", "1. Original Seed (START, 0.0) — Slower 3/4 Dance")
render_and_play(seq_custom, "2_custom_C3_seed", "2. Custom Seed (C3, 0.25) — Fast Sixteenth-Note Solo")
print("Ready to play!")

Setting up cello synthesizer (~15 seconds)...

1. Original Seed (START, 0.0) — Slower 3/4 Dance



2. Custom Seed (C3, 0.25) — Fast Sixteenth-Note Solo


Ready to play!
